In [9]:
import csv
import json
import os
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = "https://www.thebluealliance.com/api/v3"

# Inclusive year range to process.
START_YEAR = 2022
END_YEAR = 2026


def load_tba_key(env_path: str = ".env") -> str:
    key = os.getenv("TBA_KEY")
    if key:
        return key.strip()

    p = Path(env_path)
    if not p.exists():
        raise FileNotFoundError(f"Could not find {env_path} and TBA_KEY is not set in environment.")

    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("TBA_KEY="):
            value = line.split("=", 1)[1].strip().strip('"').strip("'")
            if value:
                return value

    raise ValueError("TBA_KEY was not found in .env")


def tba_get(path: str, key: str):
    url = f"{BASE_URL}{path}"
    headers = {
        "X-TBA-Auth-Key": key,
        "User-Agent": "cs2621-term-project/1.0 (+python)",
        "Accept": "application/json",
    }
    req = Request(url, headers=headers)
    try:
        with urlopen(req, timeout=30) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP error {e.code} for {url}: {body}") from e
    except URLError as e:
        raise RuntimeError(f"Network error for {url}: {e}") from e


def get_winning_alliance_name(alliances):
    for alliance in alliances or []:
        status = alliance.get("status") or {}
        if status.get("status") == "won":
            return alliance.get("name")
    return None


def normalize_alliance_row(event_key, year, alliance_obj, winner_name):
    picks = alliance_obj.get("picks") or []

    picks = [p.replace("frc", "").strip() if isinstance(p, str) else p for p in picks]

    return {
        "year": year,
        "event_key": event_key,
        "alliance_name": alliance_obj.get("name"),
        "alliance_number": alliance_obj.get("number"),
        "captain": picks[0] if len(picks) > 0 else None,
        "pick_2": picks[1] if len(picks) > 1 else None,
        "pick_3": picks[2] if len(picks) > 2 else None,
        "is_winner": alliance_obj.get("name") == winner_name,
    }


def fetch_playoff_alliances_for_year(year: int, tba_key: str):
    events = tba_get(f"/events/{year}", tba_key)
    rows = []

    for event in events:
        event_key = event.get("key")
        if not event_key:
            continue

        alliances = tba_get(f"/event/{event_key}/alliances", tba_key)
        if not alliances:
            continue

        winner_name = get_winning_alliance_name(alliances)

        for alliance in alliances:
            rows.append(
                normalize_alliance_row(
                    event_key=event_key,
                    year=year,
                    alliance_obj=alliance,
                    winner_name=winner_name,
                )
            )

    return rows


def fetch_playoff_alliances_for_years(start_year: int, end_year: int, tba_key: str):
    all_rows = []

    for year in range(start_year, end_year + 1):
        year_rows = fetch_playoff_alliances_for_year(year, tba_key)
        all_rows.extend(year_rows)
        print(f"{year}: {len(year_rows)} alliance rows")

    return all_rows


def write_csv(rows, output_path="playoff_alliances_with_winners.csv"):
    fieldnames = [
        "year",
        "event_key",
        "alliance_name",
        "alliance_number",
        "captain",
        "pick_2",
        "pick_3",
        "is_winner",
    ]

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def main():
    tba_key = load_tba_key(".env")
    rows = fetch_playoff_alliances_for_years(START_YEAR, END_YEAR, tba_key)
    write_csv(rows)

    print(f"Wrote {len(rows)} alliance rows to playoff_alliances_with_winners.csv")
    if rows:
        print("Sample row:")
        print(rows[0])


main()

2022: 1752 alliance rows
2023: 1983 alliance rows
2024: 2167 alliance rows
2025: 2446 alliance rows
2026: 1597 alliance rows
Wrote 9945 alliance rows to playoff_alliances_with_winners.csv
Sample row:
{'year': 2022, 'event_key': '2022alhu', 'alliance_name': 'Alliance 1', 'alliance_number': None, 'captain': '364', 'pick_2': '59', 'pick_3': '3959', 'is_winner': True}
